In [ ]:
import os
import pandas as pd
import psycopg2
import DATABASE_CONFIG

conn = psycopg2.connect(
    dbname=DATABASE_CONFIG.DB_NAME,
    user=DATABASE_CONFIG.DB_USER,
    password=DATABASE_CONFIG.DB_PASSWORD,
    host=DATABASE_CONFIG.DB_HOST,
    port=DATABASE_CONFIG.DB_PORT
)

cursor = conn.cursor()

In [64]:
command = """
SELECT id_cidade, nome_normalizado FROM cidade
"""
cursor.execute(command)
rows_cidade = cursor.fetchall()
ids_cidade = {nome: ids_cidade for ids_cidade, nome in rows_cidade}

print(ids_cidade)

{"alta floresta d'oeste": 672, 'alto alegre dos parecis': 673, 'alto paraiso': 674, "alvorada d'oeste": 675, 'ariquemes': 676, 'buritis': 677, 'cabixi': 678, 'cacaulandia': 679, 'cacoal': 680, 'campo novo de rondonia': 681, 'candeias do jamari': 682, 'castanheiras': 683, 'cerejeiras': 684, 'chupinguaia': 685, 'colorado do oeste': 686, 'corumbiara': 687, 'costa marques': 688, 'cujubim': 689, "espigao d'oeste": 690, 'governador jorge teixeira': 691, 'guajara-mirim': 692, 'itapua do oeste': 693, 'jaru': 694, 'ji-parana': 695, "machadinho d'oeste": 696, 'ministro andreazza': 697, 'mirante da serra': 698, 'monte negro': 699, "nova brasilandia d'oeste": 700, 'nova mamore': 701, 'nova uniao': 702, 'novo horizonte do oeste': 703, 'ouro preto do oeste': 704, 'parecis': 705, 'pimenta bueno': 706, 'pimenteiras do oeste': 707, 'porto velho': 708, 'presidente medici': 1273, 'primavera de rondonia': 710, 'rio crespo': 711, 'rolim de moura': 712, "santa luzia d'oeste": 713, "sao felipe d'oeste": 714,

In [63]:
def buscar_cidade(nome_busca):
    for nome_cadastro, id_cadastro in ids_cidade.items():
        if nome_cadastro == nome_busca:
            return id_cadastro

        nome_cadastro_lista = nome_cadastro.split()
        nome_busca_lista = nome_busca.split()
        if len(nome_cadastro_lista) != len(nome_busca_lista):
            continue

        if all(
            (parte_nome_cadastro == parte_nome_busca)
            or (
                (parte_nome_cadastro.endswith(".") or parte_nome_busca.endswith("."))
                and parte_nome_cadastro[0] == parte_nome_busca[0]
            )
            for parte_nome_cadastro, parte_nome_busca in zip(
                nome_cadastro_lista, nome_busca_lista
            )
        ):
            return id_cadastro

In [61]:
def inserir_estacao_meteorologica(cursor, dados):
    try:
        nome = dados['nome']
        latitude = dados['latitude']
        longitude = dados['longitude']
        altitude = dados['altitude']
        data_fundacao = dados['data_fundacao']
        id_cidade = dados['id_cidade']

        command = """
        INSERT INTO estacao_meteorologica (nome, latitude, longitude, altitude, data_fundacao, id_cidade)
        VALUES (%s, %s, %s, %s, %s, %s)
        ON CONFLICT (nome) DO NOTHING;
        """
        cursor.execute(command, (nome, latitude, longitude, altitude, data_fundacao, id_cidade))

        cursor.execute("SELECT id_estacao_meteorologica FROM estacao_meteorologica WHERE nome = %s LIMIT 1", (nome,))
        resultado = cursor.fetchone()
        return resultado[0] if resultado else None
    except Exception as e:
        print(f"Erro ao inserir estação {dados['nome']}: {e}")
        return None

In [60]:
from datetime import datetime


def normalizar_data(data_str, formato_padrao="%Y-%m-%d", valor_padrao="1900-01-01"):
    formatos_aceitos = (
        "%d/%m/%y",
        "%Y-%m-%d",
        "%Y/%m/%d",
        "%d/%m/%y %H:%M",
        "%Y-%m-%d %H:%M",
        "%Y/%m/%d %H:%M",
        "%d/%m/%y %H%M UTC",
        "%Y-%m-%d %H%M UTC",
        "%Y/%m/%d %H%M UTC",
    )

    data_str = data_str.strip()
    for formato in formatos_aceitos:
        try:
            return datetime.strptime(data_str, formato).strftime(formato_padrao)
        except ValueError:
            continue

    return valor_padrao

In [74]:
import math

def inserir_relatorio_meteorologia(cursor, id_estacao, caminho_arquivo):
    dados_clima = pd.read_csv(caminho_arquivo, sep=';', encoding='ISO-8859-1', skiprows=8)
    dados_clima.columns = [col.strip().upper() for col in dados_clima.columns]

    def coluna_que_contem(*termos):
        return next((
            col for col in dados_clima.columns
            if all(t in col for t in termos)
        ), None)

    colunas = {
        'data': coluna_que_contem('DATA'),
        'hora': coluna_que_contem('HORA'),
        'temp_max': coluna_que_contem('TEMPERATURA', 'MÁXIMA'),
        'temp_min': coluna_que_contem('TEMPERATURA', 'MÍNIMA'),
        'precipitacao': coluna_que_contem('PRECIPITAÇÃO'),
        'umidade': coluna_que_contem('UMIDADE'),
        'pressao': coluna_que_contem('PRESSAO ATMOSFERICA', 'HORARIA'),
        'vento': coluna_que_contem('VENTO', 'VELOCIDADE'),
    }

    def parse_valor(valor):
        valor = str(valor).strip().replace(',', '.')
        try:
            fval = float(valor)
            return None if math.isnan(fval) or valor in ['', '-9999'] else fval
        except ValueError:
            return None

    for _, linha in dados_clima.iterrows():
        try:
            data_str = str(linha[colunas['data']]).strip()
            hora_str = str(linha[colunas['hora']]).strip()
            data_hora = normalizar_data(f"{data_str} {hora_str}", formato_padrao="%Y-%m-%d %H:%M:%S")

            parametros = {
                'data_horario_coleta': data_hora,
                'temperatura_max': parse_valor(linha[colunas['temp_max']]),
                'temperatura_min': parse_valor(linha[colunas['temp_min']]),
                'precipitacao': parse_valor(linha[colunas['precipitacao']]),
                'umidade': parse_valor(linha[colunas['umidade']]),
                'pressao_atm': parse_valor(linha[colunas['pressao']]),
                'vento_velocidade': parse_valor(linha[colunas['vento']]),
                'id_estacao_meteorologica': id_estacao
            }

            comando = """
                INSERT INTO relatorio_meteorologia (
                    data_horario_coleta,
                    temperatura_max,
                    temperatura_min,
                    precipitacao,
                    umidade,
                    pressao_atm,
                    vento_velocidade,
                    id_estacao_meteorologica
                )
                VALUES (%(data_horario_coleta)s, %(temperatura_max)s, %(temperatura_min)s,
                        %(precipitacao)s, %(umidade)s, %(pressao_atm)s, %(vento_velocidade)s,
                        %(id_estacao_meteorologica)s)
            """
            cursor.execute(comando, parametros)

        except Exception as e:
            print(f"Erro ao inserir linha: {e} | Linha: {linha}")

In [58]:
import unicodedata

def normalizar(texto):
    unaccent = ''.join(
        c for c in unicodedata.normalize('NFD', texto)
        if unicodedata.category(c) != 'Mn'
    )
    return unaccent.lower()

In [57]:
import re

def normalizar_nome_cidade(nome):
    normalizar_parenteses = re.sub(r"\s*\([^)]*\)", "", nome)
    normalizar_espacos = re.sub(r"\.\s*", ". ", normalizar_parenteses)
    return normalizar(normalizar_espacos.strip())

In [80]:
for ano in range(2000, 2025):
    print("Inserindo CLIMA %s" % ano)
    caminho_pasta = "../datasets/clima%s/" % ano

    for nome_arquivo in os.listdir(caminho_pasta):
        if nome_arquivo.endswith(".CSV"):
            caminho_arquivo = os.path.join(caminho_pasta, nome_arquivo)
            try:
                dados = pd.read_csv(
                    caminho_arquivo,
                    sep=";",
                    encoding="ISO-8859-1",
                    header=None,
                    nrows=8,
                )

                dados_estacao_meteorologica = {
                    linha[0].strip(":").strip(): linha[1]
                    for _, linha in dados.iterrows()
                }
                codigo_estacao = dados_estacao_meteorologica.get(
                    "CODIGO (WMO)", ""
                ).strip()
                if not codigo_estacao[0].upper() == "A":
                    continue

                nome_estacao = (
                    dados_estacao_meteorologica.get("ESTAÇÃO")
                    or dados_estacao_meteorologica.get("ESTACAO")
                    or dados_estacao_meteorologica.get("ESTAC?O")
                )
                if nome_estacao is None:
                        print(
                            f"Erro obter nome da estação. Pulando estação."
                        )
                        continue
                nome_estacao = nome_estacao.strip()
                nome_estacao = nome_estacao.title()

                cursor.execute(
                    "SELECT id_estacao_meteorologica FROM estacao_meteorologica WHERE nome = %s LIMIT 1",
                    (nome_estacao,),
                )
                resultado_estacao = cursor.fetchone()
                if resultado_estacao:
                    inserir_relatorio_meteorologia(
                        cursor, resultado_estacao[0], caminho_arquivo
                    )
                    continue

                uf = dados_estacao_meteorologica.get("UF", "").strip()
                cursor.execute(
                    "SELECT id_estado FROM estado WHERE uf = %s LIMIT 1", (uf,)
                )
                resultado_estado = cursor.fetchone()
                if not resultado_estado:
                    continue
                id_estado = resultado_estado[0]

                nome_cidade = nome_estacao.split("-")[0].strip()
                nome_cidade = normalizar_nome_cidade(nome_cidade)
                id_cidade = buscar_cidade(nome_cidade)
                if id_cidade is None:
                    continue

                parametros_estacao_meteorologica = {
                    "nome": nome_estacao,
                    "latitude": float(
                        dados_estacao_meteorologica.get("LATITUDE", "0").replace(
                            ",", "."
                        )
                    ),
                    "longitude": float(
                        dados_estacao_meteorologica.get("LONGITUDE", "0").replace(
                            ",", "."
                        )
                    ),
                    "altitude": float(
                        dados_estacao_meteorologica.get("ALTITUDE", "0").replace(
                            ",", "."
                        )
                    ),
                    "data_fundacao": normalizar_data(
                        dados_estacao_meteorologica.get("DATA DE FUNDAÇÃO")
                        or dados_estacao_meteorologica.get(
                            "DATA DE FUNDAÇÃO (YYYY-MM-DD)"
                        )
                        or dados_estacao_meteorologica.get("DATA DE FUNDACAO")
                        or dados_estacao_meteorologica.get("DATA DE FUNDAC?O", "")
                    ),
                    "id_cidade": id_cidade,
                }

                id_estacao = inserir_estacao_meteorologica(
                    cursor, parametros_estacao_meteorologica
                )
                inserir_relatorio_meteorologia(
                    cursor, id_estacao, caminho_arquivo
                )

            except Exception as erro:
                print(f"Erro ao processar o arquivo {nome_arquivo}: {erro}")

conn.commit()


Inserindo CLIMA 2000
Inserindo CLIMA 2001
Inserindo CLIMA 2002
Inserindo CLIMA 2003
Inserindo CLIMA 2004
Inserindo CLIMA 2005
Inserindo CLIMA 2006
Inserindo CLIMA 2007
Inserindo CLIMA 2008
Inserindo CLIMA 2009
Inserindo CLIMA 2010
Inserindo CLIMA 2011
Inserindo CLIMA 2012
Inserindo CLIMA 2013
Inserindo CLIMA 2014
Inserindo CLIMA 2015
Inserindo CLIMA 2016
Inserindo CLIMA 2017
Inserindo CLIMA 2018
Inserindo CLIMA 2019
Inserindo CLIMA 2020
Inserindo CLIMA 2021
Inserindo CLIMA 2022
Inserindo CLIMA 2023
Inserindo CLIMA 2024


In [ ]:
conn.rollback()

In [78]:
cursor.close()
conn.close()
# Close the connection